In [60]:
from scipy.optimize import linprog
import pandas as pd
import pulp

In [ ]:
df = pd.read_excel("tabela.xlsx")
def exercice_query(df, exercices):
    score = []
    
    exercices_data = df[
        (df["primaryMuscles"].isin(exercices) | 
        df["secondaryMuscles"].astype(str).str.lower().apply(
            lambda x: any(ex in x for ex in exercices))) &
        df["category"].isin(["powerlifting","forca"]) &
        df["level"].isin(["iniciante"])
    ]
    
    for index, row in exercices_data.iterrows():
        x = 0
        for col in exercices:
            if col in row:
                x += row[col]
        score.append(x)
        
    # Se quiser salvar o score de volta no dataframe:
    
    print(score)
    
    return exercices_data,score

exercices,score = exercice_query(df,["panturrilhas","isquiotibiais"])
df_size = len(exercices)
nomes_exercicios = exercices["name"].to_numpy()
exercices_data = exercices.iloc[:, 15:32]
exercices_columns = exercices_data.columns.to_numpy()
exercices_data = exercices_data.to_numpy()
print(exercices_columns)
TEMPO_TREINO = 60 #MIN
DIAS_TREINO = 2
TEMPO_SERIE = 3
TEMPO_DESCANCO = 1.5
TEMPO_DE_PREPARACA0 = 10
SERIES_DISPONÍVEIS =  (int)(((TEMPO_TREINO - TEMPO_DE_PREPARACA0)  * DIAS_TREINO) / (TEMPO_SERIE + TEMPO_DESCANCO))     
print(exercices)


[1.0499999999999998, 1.0499999999999998, 0.7, 0.7, 0.7, 0.7, 0.7, 0.7, 0.35, 0.7, 0.35, 0.35, 0.7, 0.7, 0.7, 0.7, 0.7, 0.7, 0.7, 0.35, 0.35, 0.7, 0.7, 0.35, 0.7, 0.35, 0.35, 0.7, 0.35, 0.7, 0.35, 0.7, 0.7, 0.7, 0.35, 0.7, 0.7, 0.7, 0.7, 0.35, 0.35, 0.35, 0.7, 0.7, 0.7, 0.7, 0.7, 0.35, 0.7, 0.7, 0.7, 0.7, 0.35, 0.7, 0.35, 0.35, 0.35]
['abdominais' 'isquiotibiais' 'adutores' 'quadriceps' 'biceps' 'ombros'
 'peito' 'meio-das-costas' 'panturrilhas' 'gluteos' 'inferior-das-costas'
 'dorsais' 'triceps' 'trapezio' 'antebracos' 'pescoco' 'abdutores']
                                                  name   force      level  \
33                               Prancha de Equilíbrio     NaN  iniciante   
34                           Flexão de Pernas com Bola    pull  iniciante   
36                                   Bom Dia com Banda    pull  iniciante   
37                         Bom Dia com Puxada de Banda    pull  iniciante   
57           Elevação de Panturrilha Sentado com Barra    push  in

In [62]:
def simplex(exercices,score,df_size,exercices_data,exercices_columns):
    modelo = pulp.LpProblem("Otimizacao_Treino_Hipertrofia", pulp.LpMaximize)
    x = [pulp.LpVariable(f"x_{i}", lowBound=0,upBound=5, cat='Integer') for i in range(df_size)]
    y = [pulp.LpVariable(f"y_{i}", cat='Binary') for i in range(df_size)]
    modelo += pulp.lpSum(score[i] * x[i] for i in range(df_size)), "Maximizar_Score_Total"
    MAX_ESTIMULO_POR_MUSCULO = [25,20,16,20,26,26,22,25,20,16,12,18,26,14,16,16,16]
    musculo = 0

    for m in exercices_columns:
        modelo += pulp.lpSum(exercices_data[i, musculo] * x[i] for i in range(df_size)) <= MAX_ESTIMULO_POR_MUSCULO[musculo]
        musculo +=1;
        
    for i in range(df_size):
        # Se o exercício entrou (y=1), x deve ser no mínimo 3 séries
        modelo += x[i] >= 3 * y[i], f"Min_Series_{i}"
        # Se o exercício entrou, x não pode passar de 5 séries (por exemplo). Se y=0, x força a ser 0.
        modelo += x[i] <= 10 * y[i], f"Max_Series_{i}"
    #escolhe no máximo 5 exercícios
    modelo += pulp.lpSum(y[i] for i in range(df_size)) <= 6
    #limitação de séries disponíveis com base no tempo
    modelo += pulp.lpSum(x[i] for i in range(df_size)) <= SERIES_DISPONÍVEIS
    modelo.solve(pulp.PULP_CBC_CMD(msg=0, timeLimit=120))

    return x
x = simplex(exercices,score,df_size,exercices_data,exercices_columns);

In [63]:


n = 0;
for i in range(df_size):
    if x[i].varValue > 0:
        n+=1
        print(f"exercício:{nomes_exercicios[i]}\nseries: {x[i].varValue}")
        
        
print(n)

exercício:Arranco Alternado a Partir da Suspensão
series: 3.0
exercício:Elevação Glúteo-Isquiostibial no Chão
series: 4.0
exercício:Afundo com Passagem de Kettlebell
series: 5.0
exercício:Elevação de Glúteos e Isquiotibiais Natural
series: 5.0
exercício:Levantamento Terra Sumô com Banda Reversa
series: 5.0
5
